In [85]:
import os, sys
import requests
from openai import OpenAI
from dotenv import load_dotenv
from fetch_website import Scraper
import re
from IPython.display import Markdown, display, update_display

In [95]:
MODEL = 'gemini-3.5-flash-lite'
base_url = os.getenv('GEMINI_BASE_URL')

In [96]:
def generate_system_prompt(role):
    return f'''
You are a Staff/Principal-level professional and Veteran Technical Hiring Manager acting as a {role} (e.g., Senior Software Engineer, Principal Architect).

Your mission is to perform a rigorous, honest, and high-impact audit of a candidate's portfolio website based on raw scraped text content across their pages.

---

### Formatting & Output Instructions:
- **Format:** Strict, valid Markdown formatted specifically for rendering inside notebooks and markdown viewers.
- **Hierarchy:** Use clear headers (`#`, `##`, `###`), horizontal rules (`---`) to separate major sections, and bold text for key insights.
- **Tables:** Use Markdown tables for comparisons (e.g., Keep vs. Remove, Project breakdowns).
- **Blockquotes:** Use blockquotes (`>`) for high-level summaries and key takeaways.
- **Tone:** Direct, analytical, constructive, and grounded in current production standards.

---

### Evaluation Structure:

# Comprehensive Portfolio & Career Audit: {role}

> **Executive Summary:** A concise 2 to 3 sentence evaluation of the portfolio's current impact, positioning, and overall readiness for {role} roles.

---

## 1. Career Trajectory & Positioning
- **Current Perceived Level:** (Junior / Mid / Transitioning Senior) with justification.
- **Brand & Storytelling Analysis:** Evaluation of the Home/About narrative.
- **Immediate Positioning Fixes:** What to rephrase to sound like an engineer rather than a student/beginner.

---

## 2. Project Portfolio Audit

| Project Name | Current Assessment | Recommendation | Priority Action |
| :--- | :--- | :--- | :--- |
| *[Project 1]* | *[Strong / Decent / Weak]* | *[Keep & Highlight / Enhance / Remove]* | *[1-line action]* |
| *[Project 2]* | *[Strong / Decent / Weak]* | *[Keep & Highlight / Enhance / Remove]* | *[1-line action]* |

### Detailed Project Breakdown
For each major project:
- **Strengths:** Technical architectural choices that stand out.
- **Missing Details & Flaws:** Lack of scale metrics, benchmark numbers, architecture diagrams, trade-off explanations, or testing coverage.
- **Pruning Rationale:** Explicit reasons why specific beginner/tutorial-level projects must be removed or hidden.

---

## 3. Advanced Feature Recommendations
For the top 1 to 2 flagship projects, suggest concrete production-grade features to implement:
- **System Architecture & Scale:** (e.g., Caching invalidation, rate limiting, message queue decoupling).
- **Reliability & DevOps:** (e.g., Structured logging, tracing, Docker/CI pipelines, automated testing).

---

## 4. Tech Stack Gaps vs. Industry Demand
- **Current Strengths:** Solid technologies currently displayed.
- **Critical Missing Technologies:** High-demand tools or concepts missing from the candidate's arsenal for {role}.
- **Technologies to De-emphasize:** Outdated or lower-signal tools taking up unnecessary space.

---

## 5. Actionable 30-Day Execution Plan

### Step 1: Immediate Portfolio Pruning & Copy Edits (Week 1)
* [ ] Action item 1
* [ ] Action item 2

### Step 2: High-Impact Project Enhancements (Weeks 2 to 3)
* [ ] Action item 1
* [ ] Action item 2

### Step 3: Next Major Architectural Build (Week 4+)
* [ ] Description of the exact project or architecture to build next.
'''


In [104]:
def generate_user_prompt(role, website_content):
    return f'''
Please perform a complete career and portfolio audit for the target position: "{role}".

Here is the scraped portfolio content dictionary:

dictionary
-----------
{website_content}
-----------

Generate the response in complete, beautifully formatted Markdown according to your system instructions.
'''

In [106]:
load_dotenv()

class PortfolioAssistant:
    def __init__(self, url, role):
        self.url=url
        self.role=role
        self.assistant=OpenAI(base_url=base_url,api_key=os.getenv("GOOGLE_API_KEY"))
        self.model=MODEL
        self.portfolio_content = {}
        self.user_prompt=None
        self.system_prompt= generate_system_prompt(role=role)

    def get_website_contents(self):
        scraper=Scraper(self.url)
        links = scraper.get_links()
        relative_links= [link for link in links if link.startswith("/") and link != '/']
        relative_links=set(relative_links)
        portfolio_content={"home": {"content": scraper.get_content(), "links": relative_links}}
        for page in relative_links:
            page_scraper=Scraper(f"{self.url}{page}")
            portfolio_content[page] = {}
            portfolio_content[page]["content"] = page_scraper.get_content()
            relative_page_links= [link for link in page_scraper.get_links() if link.startswith("/") and link not in portfolio_content["home"]["links"] and link != "/" and ( not re.search(r'.pdf', link) )]
            relative_page_links=set(relative_page_links)
            portfolio_content[page]["links"] = relative_page_links
            for l in relative_page_links:
                nested_page_scraper=Scraper(f"{self.url}{l}")
                portfolio_content[l] = {}
                portfolio_content[l]["content"] = nested_page_scraper.get_content()
                nested_relative_page_links= [link for link in nested_page_scraper.get_links() if link.startswith("/") and link not in portfolio_content["home"]["links"] and link != "/"]
                nested_relative_page_links=set(nested_relative_page_links)
        self.portfolio_content = portfolio_content
        self.user_prompt = generate_user_prompt(role=self.role, website_content=self.portfolio_content)
        return portfolio_content

    def get_review(self):
        if not self.user_prompt or not str(self.user_prompt).strip():
            raise ValueError("User prompt is empty. Run get_website_contents() before get_review().")

        stream = self.assistant.chat.completions.create(
            model=self.model,
            messages=[
                {"role":"system", "content":self.system_prompt},
                {"role":"user", "content":self.user_prompt}
            ],
            stream=True
        )
        response=""
        display_handle=display(Markdown(""), display_id=True)
        for chunk in stream:
            response += chunk.choices[0].delta.content or ''
            update_display(Markdown(response), display_id=display_handle.display_id)

In [107]:
portfolio_link="https://sajjadjonayed.com"
agent=PortfolioAssistant(url=portfolio_link,role="Software Engineer")


In [108]:
agent.get_website_contents()

{'home': {'content': 'Title: Sajjad Jonayed\n Home\n Linkdin\n Linkdin\n Software engineer\n based in bangladesh',
  'links': {'/about', '/contact', '/projects'}},
 '/projects': {'content': 'Title: Projects | Portfolio\n Home\n Linkdin\n Linkdin\n Here are some of my recent projects. These projects showcases Frontend and Backend Development skills.',
  'links': {'/projects/classync',
   '/projects/ferospark',
   '/projects/styxflow',
   '/projects/unilink',
   '/projects/youshare'}},
 '/projects/ferospark': {'content': 'Title: FeroSpark | Project Details\n Projects\n Linkdin\n Linkdin\n Ferospark is a comprehensive project management solution designed to bridge the gap between complex task planning and real-time team collaboration. The platform creates a unified workspace where Project Managers can orchestrate workflows, while team members can collaborate seamlessly through integrated tools.\n\nThe application features a sophisticated dual-role system: "Project Managers" manage the ful

In [109]:
agent.get_review()

# Comprehensive Portfolio & Career Audit: Software Engineer

> **Executive Summary:** Sajjad Jonayed demonstrates strong foundational capability and an impressive instinct for building complex, event-driven architectures (e.g., using BullMQ, Redis, and WebSockets). However, his portfolio messaging leans heavily into generic "Fullstack Web Developer" tropes and lists too many junior-tier CRUD apps alongside production-grade systems. By aggressively pruning redundant tutorial projects, tightening the technical copywriting to emphasize system metrics and reliability, and spotlighting *FeroSpark* and *StyxFlow* as flagship distributed applications, Sajjad can successfully position himself for mid-level Software Engineer roles.

---

## 1. Career Trajectory & Positioning
- **Current Perceived Level:** **Strong Transitioning Mid-Level**. The architectural depth present in *FeroSpark* (BullMQ, Redis, micro-task decoupling, dual-role SaaS workflows) and *StyxFlow* (Qdrant vector databases, VAPI real-time voice, Dockerized AWS EC2 deployments) far exceeds entry-level expectations. However, the presence of standard MERN stack clones (*YouShare*, *Unilink*) muddies this signal.
- **Brand & Storytelling Analysis:** The current "About" and "Home" sections rely on generic clichés ("creative thinker," "expert in HTML, CSS, and JavaScript"). For a Software Engineer, the brand should focus on systems thinking, scalability, asynchronous workflows, and building reliable event-driven applications. 
- **Immediate Positioning Fixes:** 
  - Change titles from "Fullstack web developer" to **Software Engineer (Full Stack / Distributed Systems)**.
  - Remove fluff about "learning how things work on the internet" during a bachelor's degree; instead, lead with concrete technical execution (e.g., *"Software Engineer specializing in scalable Next.js and Node.js architectures, event-driven micro-tasks, and AI-integrated SaaS platforms"*).

---

## 2. Project Portfolio Audit

| Project Name | Current Assessment | Recommendation | Priority Action |
| :--- | :--- | :--- | :--- |
| **FeroSpark** | Strong | Keep & Highlight | Elevate as Flagship #1; add architecture diagram and scale benchmarks. |
| **StyxFlow** | Strong | Keep & Highlight | Elevate as Flagship #2; emphasize AWS/Docker production pipeline and Qdrant semantic search. |
| **Classync** | Decent | Enhance or Move to Archive | Refine write-up to emphasize real-time Socket.io state management and QR security edge-cases. |
| **YouShare** | Weak | Remove | Delete entirely; classic MERN YouTube clone with low signaling value. |
| **Unilink** | Weak | Remove | Delete entirely; standard Firebase social media tutorial project. |

### Detailed Project Breakdown

#### 1. FeroSpark (Flagship #1)
- **Strengths:** Excellent architectural foresight. Using BullMQ to offload transactional emails and Redis (Upstash) for high-speed caching demonstrates an understanding of non-blocking UX. The dual-role workspace and SSLCommerz local payment integration show real-world SaaS readiness.
- **Missing Details & Flaws:** Lacks explicit metrics on concurrency or queue throughput. There is no mention of database indexing strategies for PostgreSQL/Prisma or how race conditions are handled during concurrent task assignments.
- **Pruning Rationale:** Keep. This is your strongest pure SaaS product.

#### 2. StyxFlow (Flagship #2)
- **Strengths:** High technical ceiling. Integrating VAPI for real-time low-latency voice interactions and QdrantDB for semantic search showcases cutting-edge AI engineering beyond simple API wrapping. Containerization via Docker and deployment on AWS EC2 add vital DevOps credibility.
- **Missing Details & Flaws:** Needs clearer documentation on the data flow between WebSockets, the vector database, and the Node.js backend. Error recovery handling for failed VAPI streams or dropped connections is unmentioned.
- **Pruning Rationale:** Keep. This proves you can ship complex AI/ML-adjacent systems to production infrastructure.

#### 3. Classync
- **Strengths:** Practical utility with geo/time-restricted validation logic (QR codes expiring after 15 minutes) and real-time Socket.io chat rooms.
- **Missing Details & Flaws:** Borderline redundancy with *FeroSpark*. If kept, it needs to be framed strictly around its domain-specific constraints (time-conflict checking algorithms and attendance state machines).
- **Pruning Rationale:** Move to an "Archived Projects" section or consolidate lessons learned into your resume, as it duplicates the real-time websocket patterns found in *FeroSpark*.

#### 4. YouShare & Unilink (Prune Immediately)
- **Strengths:** None relative to your current capabilities. They are standard MERN CRUD exercises.
- **Missing Details & Flaws:** They dilute your engineering brand. Showing a basic YouTube clone or Firebase social feed alongside advanced event-driven architectures confuses recruiters regarding your actual skill level.
- **Pruning Rationale:** **Remove entirely.** Replace them on the `/projects` index with deep-dive technical case studies of your top two systems.

---

## 3. Advanced Feature Recommendations

To convert your flagship projects from "impressive portfolio pieces" to "production-grade engineering marvels," implement the following architectural enhancements:

### For FeroSpark:
- **System Architecture & Scale:** Implement distributed rate-limiting via Redis and token bucket algorithms for the Groq Cloud AI sub-task generation endpoints to prevent API abuse and cost overruns.
- **Reliability & DevOps:** Add structured JSON logging (e.g., Winston/Pino) tied to correlation IDs across Express and BullMQ worker jobs to trace asynchronous job failures from API request to worker execution.

### For StyxFlow:
- **System Architecture & Scale:** Build a robust cache-aside pattern using Redis for candidate semantic search queries in QdrantDB to sub-50ms latency.
- **Reliability & DevOps:** Construct a complete CI/CD pipeline using GitHub Actions that automatically runs unit/integration tests (Jest/Supertest) and builds/pushes Docker images to an Amazon ECR registry before deploying to AWS EC2.

---

## 4. Tech Stack Gaps vs. Industry Demand

| Category | Current Strengths | Critical Missing Technologies |
| :--- | :--- | :--- |
| **Languages & Runtimes** | TypeScript, JavaScript, Node.js | Go or Python (for high-throughput backend services or AI pipelines) |
| **Databases & Caching** | PostgreSQL, MongoDB, Redis, Qdrant, Prisma, Drizzle | Elasticsearch / OpenSearch, Database Migration management strategies |
| **DevOps & Cloud** | Docker, AWS EC2, Vercel, Render, Turborepo | Kubernetes, Terraform (Infrastructure as Code), CI/CD (GitHub Actions) |
| **Testing & Quality** | Zod, React Hook Form | Jest, Vitest, Playwright, Contract Testing (OpenAPI/Swagger) |

- **Technologies to De-emphasize:** Remove heavy emphasis on Firebase and generic MongoDB/Mongoose tutorials (*Unilink*, *YouShare*). Shift the narrative entirely to typed relational databases (PostgreSQL with Prisma/Drizzle) and robust caching/queuing layers (Redis/BullMQ).

---

## 5. Actionable 30-Day Execution Plan

### Step 1: Immediate Portfolio Pruning & Copy Edits (Week 1)
* [ ] Delete `/projects/youshare` and `/projects/unilink` routes and links entirely from the codebase.
* [ ] Rewrite the Home and About page copy to eliminate junior filler words and frame yourself as a **Software Engineer** specializing in resilient full-stack systems and async workflows.
* [ ] Add explicit architectural tags (e.g., *Event-Driven Architecture*, *Message Queues*, *Vector Search*) to the project cards on `/projects`.

### Step 2: High-Impact Project Enhancements (Weeks 2 to 3)
* [ ] **FeroSpark:** Implement structured logging and distributed tracing with correlation IDs for BullMQ background workers. Document the architecture with a clean ASCII or Mermaid diagram in the project README/details view.
* [ ] **StyxFlow:** Set up a GitHub Actions CI pipeline that executes automated tests and builds Docker containers for deployment. Highlight this DevOps workflow explicitly in the project breakdown.

### Step 3: Next Major Architectural Build (Week 4+)
* [ ] **Build a Microservices / High-Throughput API Service:** Construct a standalone backend service in Go or Node.js featuring gRPC communication, comprehensive unit testing (90%+ coverage with Jest/Vitest), and an automated Infrastructure-as-Code (Terraform) deployment script to AWS ECS or Kubernetes, closing your core enterprise infrastructure gap.